In [1]:
import pandas as pd
from sqlalchemy import create_engine
import csv
import io

# Ruta al archivo limpio
DATA_PATH = '../data/'
cleaned_file = DATA_PATH + 'ecommerce_cleaned_for_cohorts.csv'

# Conexión con SQLAlchemy
db_user = 'postgres'
db_password = 'admin123'
db_host = 'localhost'
db_port = '5432'
db_name = 'marketing_cohort_db'

engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')
conn = engine.raw_connection()

# Usar COPY para carga masiva (muy rápido)
print("Cargando datos con COPY (esto será mucho más rápido)...")
with open(cleaned_file, 'r', encoding='utf-8') as f:
    cursor = conn.cursor()
    # Reemplazar la tabla si existe
    cursor.execute("DROP TABLE IF EXISTS transactions;")
    # Crear tabla automáticamente desde el CSV (requiere crear schema)
    df_sample = pd.read_csv(cleaned_file, nrows=1)
    columns = df_sample.columns.tolist()
    create_stmt = f"CREATE TABLE transactions ({', '.join([f'\"{col}\" TEXT' for col in columns])});"
    cursor.execute(create_stmt)
    # Copiar datos
    cursor.copy_expert(f"COPY transactions FROM STDIN WITH CSV HEADER DELIMITER ','", f)
    conn.commit()
    cursor.close()

print("¡Carga completada exitosamente en menos de 1 minuto!")

Cargando datos con COPY (esto será mucho más rápido)...
¡Carga completada exitosamente en menos de 1 minuto!
